# Training: Loss Function Comparison (MSE vs Neg-Pearson)

**Scope:**

- **Subject:** 2
- **Channel:** Fz
- **Time window:** 1 s
- **CV:** GroupKFold, 5 folds over 40 trials
- **Train overlap:** 0% | **Test overlap:** 0%

**Settings**

- 20 epochs, batch size 16
- Adam: lr = 1e-4, weight_decay = 1e-5
- LR decayed by 0.1 every 10 epochs
- Two identical networks, only the loss differs

**Runs:**

- MSE — exp start 131
- Neg-Pearson — exp start 136

**Output directory:** `Loss-Functions/Results/`

**Model directory:** `Loss-Functions/Models/`

In [ ]:
# ============================================================
# 0. Environment setup (Colab or local)
# ============================================================
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    try:
        get_ipython().run_line_magic('matplotlib', 'inline')
    except Exception:
        pass
    !pip install -q pims


# ============================================================
# 1. Standard library
# ============================================================
import os
import sys
import pickle
import traceback

# ============================================================
# 2. Third-party
# ============================================================
import numpy as np
import torch
import torch.nn as nn

# ============================================================
# 3. Project paths
# ============================================================
if IN_COLAB:
    SOURCE_ROOT = '/content/drive/MyDrive/Colab Notebooks/Face2EEG/Source'
    DATA_ROOT   = '/content/drive/MyDrive/Colab Notebooks/Face2EEG/Data'
else:
    SOURCE_ROOT = r'D:\Thesis\Codes\Source'
    DATA_ROOT   = r'D:\Thesis\Codes\Data'

# sanity check
if not os.path.isdir(os.path.join(SOURCE_ROOT, 'face2eeg')):
    raise FileNotFoundError(
        f"face2eeg package not found under SOURCE_ROOT = {SOURCE_ROOT}. "
        f"Expected directory: {os.path.join(SOURCE_ROOT, 'face2eeg')}"
    )

if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)


# ============================================================
# 4. Package imports
# ============================================================
from face2eeg.core.experiments import run_kfold_pipeline
from face2eeg.core.models      import try_gpu, Neg_Pearson

# Framework constants (from config)
from face2eeg.core.config import (
    SEED,
    FS_EEG,
    FS_VIDEO,
)


# ============================================================
# 5. Reproducibility
# ============================================================
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# ============================================================
# Configuration
# ============================================================
BASE = os.path.join(DATA_ROOT, 'Experiments', 'Cross-Validation')

EXP_DIR     = os.path.join(BASE, 'Loss-Functions')
RESULTS_DIR = os.path.join(EXP_DIR, 'Results')
MODELS_DIR  = os.path.join(EXP_DIR, 'Models')

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)


# ============================================================
# Experiment scope
# ============================================================
SUBJECT = 2
CHANNEL = 'Fz'


# ============================================================
# Hyperparameters 
# ============================================================
TIME_WINDOW  = 1
FRAME_SIZE   = (32, 32)

EPOCHS       = 20
BATCH_SIZE   = 16
LR           = 1e-4
WEIGHT_DECAY = 1e-5
SCHED_STEP   = 10
SCHED_GAMMA  = 0.1

# Data generation (no overlap for this experiment)
TRAIN_OVERLAP       = False
TRAIN_OVERLAP_RATIO = 0.0
TEST_OVERLAP        = False
TEST_OVERLAP_RATIO  = 0.0

# Unique exp_start per loss variant
EXP_START_MSE = 131
EXP_START_NEG = 136


# ============================================================
# Device
# ============================================================
device = try_gpu()


# ============================================================
# Sanity check
# ============================================================
print("EXP_DIR     :", EXP_DIR)
print("RESULTS_DIR :", RESULTS_DIR)
print("MODELS_DIR  :", MODELS_DIR)
print()
print("SUBJECT     :", SUBJECT)
print("CHANNEL     :", CHANNEL)
print("TIME_WINDOW :", TIME_WINDOW)
print("FRAME_SIZE  :", FRAME_SIZE)
print()
print("EPOCHS      :", EPOCHS)
print("BATCH_SIZE  :", BATCH_SIZE)
print("LR          :", LR)
print("WEIGHT_DECAY:", WEIGHT_DECAY)
print("SCHED_STEP  :", SCHED_STEP)
print("SCHED_GAMMA :", SCHED_GAMMA)
print()
print("FS_EEG      :", FS_EEG, " (from config)")
print("FS_VIDEO    :", FS_VIDEO, " (from config)")
print("device      :", device)

## Run 1 — MSE loss

Trains a PhysNet-3DCNN-ED with MSE loss on the 5-fold CV splits.

In [ ]:
result_path = os.path.join(
    RESULTS_DIR, f'sub{SUBJECT:02d}_ch{CHANNEL}_MSE.pkl'
)

if os.path.isfile(result_path):
    print(f"[skip] {result_path} already exists")
else:
    print(f"\n{'='*60}")
    print(f"  MSE — Subject {SUBJECT} | Channel {CHANNEL} | T={TIME_WINDOW}s")
    print(f"{'='*60}")

    try:
        results_mse = run_kfold_pipeline(
            subject_id=SUBJECT,
            channel_name=CHANNEL,
            time_window=TIME_WINDOW,

            # CV splits
            n_points=40, n_splits=5,

            # scaler
            scaler_method='z-score',

            # data generation
            frame_size=FRAME_SIZE,
            video_fr=FS_VIDEO,
            signal_fs=FS_EEG,
            y_type='RawEEG',

            # train
            train_overlap=TRAIN_OVERLAP,
            train_overlap_ratio=TRAIN_OVERLAP_RATIO,
            train_shuffle_output=True,
            train_signal_normalize=True,

            # test
            test_overlap=TEST_OVERLAP,
            test_overlap_ratio=TEST_OVERLAP_RATIO,
            test_shuffle_output=False,
            test_signal_normalize=True,

            # model
            video_length=None,
            signal_length=None,

            # training
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            learning_rate=LR,
            weight_decay=WEIGHT_DECAY,
            loss_fn=nn.MSELoss(),
            scheduler_step_size=SCHED_STEP,
            scheduler_gamma=SCHED_GAMMA,

            # prediction
            predict_batch_size=BATCH_SIZE,

            # output
            model_exp_start=EXP_START_MSE,
            save_dir=MODELS_DIR,
            device=device,
            return_history=True,
            verbose=True,
            root_dir=DATA_ROOT,
        )

        with open(result_path, 'wb') as f:
            pickle.dump(results_mse, f)
        print(f"\n[saved] {result_path}")

    except Exception as e:
        print(f"\n!! ERROR for MSE: {e}")
        traceback.print_exc()

## Run 2 — Neg-Pearson loss

Trains an identical PhysNet-3DCNN-ED with the negative Pearson
correlation loss on the same 5-fold CV splits.

In [ ]:
result_path = os.path.join(
    RESULTS_DIR, f'sub{SUBJECT:02d}_ch{CHANNEL}_NegPearson.pkl'
)

if os.path.isfile(result_path):
    print(f"[skip] {result_path} already exists")
else:
    print(f"\n{'='*60}")
    print(f"  Neg-Pearson — Subject {SUBJECT} | Channel {CHANNEL} | T={TIME_WINDOW}s")
    print(f"{'='*60}")

    try:
        results_neg = run_kfold_pipeline(
            subject_id=SUBJECT,
            channel_name=CHANNEL,
            time_window=TIME_WINDOW,

            # CV splits
            n_points=40, n_splits=5,

            # scaler
            scaler_method='z-score',

            # data generation
            frame_size=FRAME_SIZE,
            video_fr=FS_VIDEO,
            signal_fs=FS_EEG,
            y_type='RawEEG',

            # train
            train_overlap=TRAIN_OVERLAP,
            train_overlap_ratio=TRAIN_OVERLAP_RATIO,
            train_shuffle_output=True,
            train_signal_normalize=True,

            # test
            test_overlap=TEST_OVERLAP,
            test_overlap_ratio=TEST_OVERLAP_RATIO,
            test_shuffle_output=False,
            test_signal_normalize=True,

            # model
            video_length=None,
            signal_length=None,

            # training
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            learning_rate=LR,
            weight_decay=WEIGHT_DECAY,
            loss_fn=Neg_Pearson(),
            scheduler_step_size=SCHED_STEP,
            scheduler_gamma=SCHED_GAMMA,

            # prediction
            predict_batch_size=BATCH_SIZE,

            # output
            model_exp_start=EXP_START_NEG,
            save_dir=MODELS_DIR,
            device=device,
            return_history=True,
            verbose=True,
            root_dir=DATA_ROOT,
        )

        with open(result_path, 'wb') as f:
            pickle.dump(results_neg, f)
        print(f"\n[saved] {result_path}")

    except Exception as e:
        print(f"\n!! ERROR for Neg-Pearson: {e}")
        traceback.print_exc()